# Controlled Descent Simulator — Rocket Model for Nonlinear MPC (iLQR)

This notebook derives the **prediction model** a nonlinear Model-Predictive
Controller (MPC) needs to fly the 6-DOF rocket, and exports it to C++ for the
runtime. It is the *model half* of the split described in the repository: the
vehicle dynamics live here, while the **control algorithm** — the
control-limited iLQR / DDP solver — is derived and validated once, vehicle-
agnostically, in [`../control/ilqr.ipynb`](../control/ilqr.ipynb) and shipped as
`libs/control/ilqr.hpp`. We *use* that solver here through its Python mirror
`../control/ilqr_ref.py`; we do **not** re-derive it.

**What "MPC" means for us.** At each control step we look a fixed number of
steps `N` into the future (the *horizon*), and choose the command sequence that
minimises a tracking cost while respecting the actuator limits — then we apply
only the first command and repeat next step. This is *receding-horizon* control.
The optimiser is **iLQR** (iterative LQR): it repeatedly builds a local
linear-quadratic model of the problem and solves that, converging to the
constrained optimum. Every term above is defined where it is first used.

**How the rocket differs from the quadrotor MPC** (its sibling notebook
[`dynamics_quadRotor_MPC01.ipynb`](dynamics_quadRotor_MPC01.ipynb)) — and why it
is *simpler*:

| | quadrotor MPC | **rocket MPC (this notebook)** |
|---|---|---|
| attitude | unit **quaternion** (13 states) | **Euler angles** (12 states) |
| attitude error | error-quaternion (small-angle vec) | **direct angle differences** |
| state projection | quaternion renormalisation each step | **none** (Euler needs no projection) |
| actuators | 4 motor thrusts + allocation | **direct wrench** `[F1,T1,T2,T3]` |
| control box | one uniform thrust box | **per-input box** (thrust ≠ torque scales) |

Everything downstream — the solver, the export path, the C++ runtime seam — is
shared with the quadrotor MPC; only the two model-specific pieces (the dynamics
and the tracking cost) change.

*References.* Rocket rigid-body dynamics: Stevens, Lewis & Johnson, *Aircraft
Control and Simulation*, 3rd ed. (2015). The Euler-angle 6-DOF model reuses the
derivation in [`dynamics_rocket_FFLQR01.ipynb`](dynamics_rocket_FFLQR01.ipynb).
iLQR / control-limited DDP: Tassa, Erez & Todorov, *Control-limited differential
dynamic programming* (ICRA 2014); the full derivation is in `../control/ilqr.ipynb`.

In [ ]:
import numpy as np
import sympy as sp
from sympy import symbols, Matrix, cos, sin, simplify
import matplotlib.pyplot as plt
sp.init_printing(use_latex='mathjax', wrap_line=False)
print("SymPy:", sp.__version__, " NumPy:", np.__version__)

## 1. Continuous nonlinear 6-DOF model

We model the rocket as a rigid body with mass $m$ and a **diagonal** inertia
$\mathrm{diag}(I_x,I_y,I_z)$. Its state is the physical 6-DOF state — position,
attitude, and their rates — **with no controller integrators** (the MPC has no
augmented states; it re-optimises from the true state each step):

$$x=\big[\,\underbrace{x,y,z}_{\text{position}},\ \underbrace{\alpha,\beta,\psi}_{\text{Euler angles}},\ \underbrace{v_x,v_y,v_z}_{\text{velocity}},\ \underbrace{\dot\alpha,\dot\beta,\dot\psi}_{\text{angular rates}}\,\big]\in\mathbb{R}^{12}.$$

The control is the **wrench** applied directly: main thrust $F_1$ along the body
$z$-axis and three body torques $T_1,T_2,T_3$,
$u=[F_1,T_1,T_2,T_3]\in\mathbb{R}^4$. Unlike the quadrotor there is **no control
allocation** — the optimiser commands the wrench, and the actuator box acts on
it directly.

**Attitude convention.** The body-to-world rotation is the Euler sequence
$R_m(\alpha,\beta,\psi)=R_y(\alpha)\,R_x(\beta)\,R_z(\psi)$: a pitch $\alpha$
about $Y$, then a roll $\beta$ about $X$, then a yaw/spin $\psi$ about $Z$. The
thrust points along body $z$, so in the world frame it is
$R_m\,[0,0,F_1]^{\top}$. A key consequence, used below: the **innermost** $R_z$
leaves the body $z$-axis invariant, so the thrust direction — and the isotropic
lateral drag — **do not depend on $\psi$**. The rocket is roll-invariant; $\psi$
is steered independently by the yaw torque $T_3$.

### 1.1 The rotation, forces, and the assembled dynamics

We build $R_m$, then the three world-frame forces acting on the body:

- **thrust** $F_{\text{thrust}}=R_m[0,0,F_1]^{\top}$ (body-$z$, rotated to world);
- **gravity** $F_g=[0,0,-mg]^{\top}$;
- **drag** — a linear, body-axis model with lateral coefficient $c$ and axial
  coefficient $c_z$: rotate velocity into the body frame, scale by
  $-\mathrm{diag}(c,c,c_z)$, rotate back;
- **external force** $F_{\text{ext}}=[u_{Fx},u_{Fy},u_{Fz}]^{\top}$ — the
  world-frame disturbance/user push. It enters additively (so it never appears in
  the Jacobians), and the MPC predicts with $F_{\text{ext}}=0$ while the true
  plant feels it — exactly the quadrotor's convention.

Newton–Euler then gives $\dot v = (F_{\text{thrust}}+F_g+F_{\text{drag}}+F_{\text{ext}})/m$
and, with a diagonal inertia and torques as direct inputs,
$\ddot\alpha=T_1/I_x,\ \ddot\beta=T_2/I_y,\ \ddot\psi=T_3/I_z$. (As in the
FF-LQR model we use the small-angle Euler-rate ≈ body-rate simplification and
neglect the $\omega\times I\omega$ gyroscopic coupling — see that notebook's §1.)

In [ ]:
# ---- symbols (plain, so codegen turns them straight into C++ locals) ----
x, y, z            = symbols('x y z', real=True)
alpha, beta, psi   = symbols('alpha beta psi', real=True)
vx, vy, vz         = symbols('vx vy vz', real=True)
dalpha, dbeta, dpsi = symbols('dalpha dbeta dpsi', real=True)          # Euler-angle rates
F1, T1, T2, T3     = symbols('F1 T1 T2 T3', real=True)                 # wrench
uFx, uFy, uFz      = symbols('uFx uFy uFz', real=True)                 # world-frame external force
m, Ix, Iy, Iz, g   = symbols('m Ix Iy Iz g', positive=True)
c, cz              = symbols('c cz', positive=True)                    # drag coefficients

# ---- body -> world rotation  Rm = Ry(alpha) Rx(beta) Rz(psi) ----
Ry = Matrix([[ cos(alpha), 0, sin(alpha)], [0, 1, 0], [-sin(alpha), 0, cos(alpha)]])
Rx = Matrix([[1, 0, 0], [0, cos(beta), -sin(beta)], [0, sin(beta), cos(beta)]])
Rz = Matrix([[cos(psi), -sin(psi), 0], [sin(psi), cos(psi), 0], [0, 0, 1]])
Rm = simplify(Ry * Rx * Rz)

# ---- world-frame forces ----
F_thrust = Rm * Matrix([0, 0, F1])
F_grav   = Matrix([0, 0, -m * g])
vel_body = Rm.T * Matrix([vx, vy, vz])
F_drag   = Rm * (-Matrix([c, c, cz]).multiply_elementwise(vel_body))
F_ext    = Matrix([uFx, uFy, uFz])

acc = (F_thrust + F_grav + F_drag + F_ext) / m

# ---- assembled continuous model  dx/dt = f(x,u) (+ additive F_ext) ----
state = Matrix([x, y, z, alpha, beta, psi, vx, vy, vz, dalpha, dbeta, dpsi])
ctrl  = Matrix([F1, T1, T2, T3])
f = Matrix([vx, vy, vz,                 # position kinematics
            dalpha, dbeta, dpsi,        # attitude kinematics
            acc[0], acc[1], acc[2],     # translational dynamics
            T1/Ix, T2/Iy, T3/Iz])       # rotational dynamics
print("state dim:", state.shape[0], " | input dim:", ctrl.shape[0], " | f dim:", f.shape[0])

### 1.2 Geometry — thrust, tilt, and roll-invariance

The diagram (drawn on a white panel so it reads in either colour theme) shows the
body $z$-axis tilted from world-up by the pitch/roll angles $\alpha,\beta$; the
thrust rides along that axis. Spinning the body about its own $z$ (the yaw/roll
$\psi$) slides the thrust nowhere — which is why $\psi$ drops out of the
translational dynamics.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 520 260" width="520" height="260">
  <rect x="0" y="0" width="520" height="260" fill="#ffffff"/>
  <!-- world axes -->
  <g stroke="#333" stroke-width="1.5" fill="#333" font-family="sans-serif" font-size="13">
    <line x1="90" y1="220" x2="90" y2="40"/>
    <polygon points="90,34 86,46 94,46"/>
    <text x="76" y="40">z (up)</text>
    <line x1="90" y1="220" x2="270" y2="220"/>
    <polygon points="276,220 264,216 264,224"/>
    <text x="250" y="240">x</text>
  </g>
  <!-- body z-axis, tilted -->
  <g stroke="#c0392b" stroke-width="2.5" fill="#c0392b" font-family="sans-serif" font-size="13">
    <line x1="90" y1="220" x2="200" y2="70"/>
    <polygon points="204,64 190,72 199,80"/>
    <text x="205" y="66">body z  (thrust F1)</text>
  </g>
  <!-- tilt angle arc -->
  <path d="M 90 130 A 90 90 0 0 1 133 78" fill="none" stroke="#2c3e50" stroke-width="1.4"/>
  <text x="112" y="112" fill="#2c3e50" font-family="sans-serif" font-size="13">&#945;,&#946;</text>
  <!-- spin arrow about body z -->
  <g stroke="#2980b9" stroke-width="1.8" fill="none">
    <path d="M 172 96 a 16 8 -55 1 1 22 14"/>
  </g>
  <polygon points="194,110 188,104 197,102" fill="#2980b9"/>
  <text x="196" y="128" fill="#2980b9" font-family="sans-serif" font-size="13">&#968; (roll about thrust axis)</text>
  <text x="300" y="176" fill="#555" font-family="sans-serif" font-size="12">translational dynamics depend on &#945;,&#946; only —</text>
  <text x="300" y="194" fill="#555" font-family="sans-serif" font-size="12">not on &#968;: the rocket is roll-invariant.</text>
</svg>

### 1.3 Sanity check — hover is an equilibrium

A state is an *equilibrium* when $\dot x=f(x,u)=0$. Upright ($\alpha=\beta=\psi=0$),
at rest, the only way to cancel gravity is a thrust $F_1=mg$ with zero torque. We
`lambdify` $f$ into a fast numeric function and confirm it vanishes there. We use
the runtime's default rocket parameters so the numbers match the C++.

In [ ]:
params = {m: 10.0, Ix: 10.0/3.0, Iy: 10.0/3.0, Iz: 1.0, g: 9.81, c: 1.0, cz: 0.02}
m_val, g_val = params[m], params[g]

args   = (x, y, z, alpha, beta, psi, vx, vy, vz, dalpha, dbeta, dpsi, F1, T1, T2, T3)  # Jacobian args
args_f = args + (uFx, uFy, uFz)                                                        # + external force
f_l = sp.lambdify(args_f, f.subs(params), 'numpy')
def f_num(xv, uv, uF=(0.0, 0.0, 0.0)):
    return np.asarray(f_l(*xv, *uv, *uF), float).flatten()

F1_hover = m_val * g_val
x_hover = np.zeros(12)
u_hover = np.array([F1_hover, 0.0, 0.0, 0.0])
print("hover thrust F1 [N]:", round(F1_hover, 4))
print("f at hover:", np.round(f_num(x_hover, u_hover), 12))
assert np.allclose(f_num(x_hover, u_hover), 0, atol=1e-9), "hover is not an equilibrium!"
print("OK - hover is an equilibrium.")

## 2. Discretization — the prediction map $F_d$

The optimiser reasons over *discrete* steps of length `DT_MPC`. We advance the
continuous $f$ with one **RK4** (4th-order Runge–Kutta) step — the same
integrator the runtime uses. This map $F_d$ is what the MPC rolls forward to
predict the future. Unlike the quadrotor there is **no projection step**: Euler
angles need no renormalisation (a quaternion would; that is the quadrotor's
`normalize_quat`). `DT_MPC` is an MPC knob — it is both the prediction step and
the control period — tuned later; here it just discretises the model.

In [ ]:
DT_MPC = 0.02   # control / prediction period [s] (50 Hz) -- an MPC knob

def rk4_step(xv, uv, dt=DT_MPC):
    k1 = f_num(xv, uv)
    k2 = f_num(xv + 0.5*dt*k1, uv)
    k3 = f_num(xv + 0.5*dt*k2, uv)
    k4 = f_num(xv + dt*k3, uv)
    return xv + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def project(xv):        # Euler state: nothing to project (kept for API symmetry with the quad)
    return xv

def F_d(xv, uv, dt=DT_MPC):
    return project(rk4_step(xv, uv, dt))

# one predicted step from hover with a small pitch torque -> the nose should start to move
u_test = u_hover + np.array([0.0, 0.5, 0.0, 0.0])
print("x_{k+1} - x_k =", np.round(F_d(x_hover, u_test) - x_hover, 6))

## 3. Jacobians of the model — $f_x$ and $f_u$

iLQR needs a *linear* model of the dynamics at each point: the partial
derivatives $f_x=\partial f/\partial x$ (12×12) and $f_u=\partial f/\partial u$
(12×4). Because the external force enters additively, it drops out of both — so
the Jacobians are functions of $(x,u)$ only. SymPy differentiates $f$ exactly;
the solver later chain-rules these *continuous* Jacobians through the RK4 step to
get the *discrete* $A_k=\partial F_d/\partial x,\ B_k=\partial F_d/\partial u$
(this RK4 *sensitivity* propagation lives in the shared solver, `../control/`).

In [ ]:
fx = f.jacobian(state)   # 12 x 12
fu = f.jacobian(ctrl)    # 12 x 4
fx_l = sp.lambdify(args, fx.subs(params), 'numpy')
fu_l = sp.lambdify(args, fu.subs(params), 'numpy')
def fx_num(xv, uv): return np.asarray(fx_l(*xv, *uv), float)
def fu_num(xv, uv): return np.asarray(fu_l(*xv, *uv), float)
print("f_x shape:", fx.shape, " f_u shape:", fu.shape)
# roll-invariance, made concrete: no velocity row depends on psi (column 5 of the
# translational block is zero).
psi_col = fx_num(np.array([0.3,-0.2,0.1, 0.2,0.15,0.7, 0.4,-0.3,0.2, 0.1,0.0,0.05]), u_hover)[6:9, 5]
print("d(v_dot)/d(psi) =", np.round(psi_col, 12), " -> psi does not drive translation")

### 3.1 Acid test — analytic Jacobians vs finite differences

We check the analytic $f_x,f_u$ against central finite differences of $f$ at
**many** random points (not just one), and report the worst mismatch. Agreement
to ~$10^{-7}$ (the finite-difference floor) certifies the derivatives.

In [ ]:
rng = np.random.default_rng(0)
def fd_jac(xv, uv, eps=1e-6):
    A = np.zeros((12, 12)); B = np.zeros((12, 4))
    for j in range(12):
        dx = np.zeros(12); dx[j] = eps
        A[:, j] = (f_num(xv+dx, uv) - f_num(xv-dx, uv)) / (2*eps)
    for j in range(4):
        du = np.zeros(4); du[j] = eps
        B[:, j] = (f_num(xv, uv+du) - f_num(xv, uv-du)) / (2*eps)
    return A, B

worst = 0.0
for _ in range(200):
    xv = rng.normal(0, 0.5, 12); uv = u_hover + rng.normal(0, 2.0, 4)
    A, B = fx_num(xv, uv), fu_num(xv, uv)
    Afd, Bfd = fd_jac(xv, uv)
    worst = max(worst, np.max(np.abs(A-Afd)), np.max(np.abs(B-Bfd)))
print(f"worst |analytic - finite-diff| over 200 points = {worst:.2e}")
assert worst < 1e-6, "Jacobian mismatch"
print("OK - analytic Jacobians match finite differences.")

## 4. The tracking cost

The MPC minimises a sum of **squared residuals** — a Gauss–Newton (GN) cost. A
residual $r(x,u)$ is a vector that should be zero when we track perfectly; the
stage cost is $\ell=\tfrac12\lVert r\rVert^2$. Writing it this way hands the
solver a **positive-semidefinite** Hessian for free: with $J=\partial r/\partial
(x,u)$, the GN gradient is $J^{\top}r$ and the GN Hessian is $J^{\top}J\succeq0$
(we drop the second-order term), so the backward pass never has to fix an
indefinite curvature.

Our residual stacks four weighted blocks, plus the control block:

$$r_{\text{state}}=\big[\,w_p(p-p_{\text{ref}}),\ w_q(\theta-\theta_{\text{ref}}),\ w_v(v-v_{\text{ref}}),\ w_w(\dot\theta-\dot\theta_{\text{ref}})\,\big],\qquad r_{\text{ctrl}}=\big[\,r_{F_1}(F_1-F_1^{\text{ref}}),\ r_{T_1}(\cdot),\ r_{T_2}(\cdot),\ r_{T_3}(\cdot)\,\big].$$

Two rocket-specific points:

- **Attitude is a plain difference.** With Euler angles the attitude error is
  just $\theta-\theta_{\text{ref}}=[\alpha-\alpha_{\text{ref}},\ \beta-\beta_{\text{ref}},\ \psi-\psi_{\text{ref}}]$
  — no error-quaternion, no small-angle map. The heading error is **wrapped** to
  $[-\pi,\pi]$ (via $\operatorname{atan2}(\sin,\cos)$) so that, e.g., $+179^\circ$
  and $-181^\circ$ mean the same thing. Wrapping changes the residual *value* but
  not its derivative (slope $1$ almost everywhere), so the GN Jacobian $J$ stays a
  constant diagonal.
- **Per-input control weights.** The thrust $F_1$ (order $10^2$ N) and the torques
  (order $10^0$ N·m) live on very different scales, so a single control weight
  would let $F_1$ dominate. We give each input its own weight
  $r_{F_1},r_{T_1},r_{T_2},r_{T_3}$ — the diagonal of $\sqrt{R}$. (The quadrotor,
  with four identical rotors, uses one scalar.)

The weights below are the $\sqrt{}$ of the usual $Q,R$ diagonals; the values are
provisional and get exercised in the closed-loop step. In the runtime they are
exposed as tunable controller parameters.

In [ ]:
# reference symbols: position, attitude, velocity, rate, control
prx, pry, prz       = symbols('prx pry prz', real=True)
arA, arB, arP       = symbols('arA arB arP', real=True)     # attitude ref (alpha,beta,psi)
vrx, vry, vrz       = symbols('vrx vry vrz', real=True)
drA, drB, drP       = symbols('drA drB drP', real=True)     # rate ref
ur1, ur2, ur3, ur4  = symbols('ur1 ur2 ur3 ur4', real=True) # control ref (wrench)

# per-block state weights (sqrt of Q diagonal) and per-input control weights (sqrt
# of R diagonal). Tuned in the closed-loop step below; these are the values baked
# into the runtime (core/Models/RocketMPC) as the default controller parameters.
w_p, w_q, w_v, w_w      = 6.0, 8.0, 1.0, 0.50
rF1, rT1, rT2, rT3      = 0.02, 0.2, 0.2, 0.2

r_state = sp.Matrix([
    w_p*(x-prx),  w_p*(y-pry),  w_p*(z-prz),
    w_q*(alpha-arA), w_q*(beta-arB), w_q*(psi-arP),        # attitude: direct angle error (psi wrapped numerically)
    w_v*(vx-vrx), w_v*(vy-vry), w_v*(vz-vrz),
    w_w*(dalpha-drA), w_w*(dbeta-drB), w_w*(dpsi-drP)])
r_ctrl = sp.Matrix([rF1*(F1-ur1), rT1*(T1-ur2), rT2*(T2-ur3), rT3*(T3-ur4)])

J_x = r_state.jacobian(state)   # 12 x 12, constant diagonal
J_u = r_ctrl.jacobian(ctrl)     # 4 x 4,  constant diagonal
print("state residual dim:", r_state.shape[0], " | control residual dim:", r_ctrl.shape[0])

ref_syms = (prx,pry,prz, arA,arB,arP, vrx,vry,vrz, drA,drB,drP, ur1,ur2,ur3,ur4)
all_syms = (*state, *ctrl, *ref_syms)
L_rs = sp.lambdify(all_syms, r_state, 'numpy')
L_rc = sp.lambdify(all_syms, r_ctrl, 'numpy')
Jx_const = np.asarray(J_x, float)     # constant -> evaluate once
Ju_const = np.asarray(J_u, float)
IPSI = 5                               # index of the heading residual (psi)

def _wrap(a): return np.arctan2(np.sin(a), np.cos(a))
def _residuals(xv, uv, ref):
    rs = np.asarray(L_rs(*xv, *uv, *ref), float).flatten()
    rs[IPSI] = w_q * _wrap(xv[5] - ref[5])            # heading error wrapped to [-pi, pi]
    rc = np.asarray(L_rc(*xv, *uv, *ref), float).flatten()
    return rs, rc

def cost(xv, uv, ref):     rs, rc = _residuals(xv, uv, ref); return 0.5*(rs@rs + rc@rc)
def cost_lx(xv, uv, ref):  rs, _  = _residuals(xv, uv, ref); return Jx_const.T @ rs
def cost_lu(xv, uv, ref):  _, rc  = _residuals(xv, uv, ref); return Ju_const.T @ rc
def cost_Jx(xv, uv, ref):  return Jx_const
def cost_Ju(xv, uv, ref):  return Ju_const

### 4.1 The terminal cost, and why a finite horizon needs one

A finite horizon is short-sighted: minimising cost only up to step $N$ invites
the optimiser to arrive at $N$ with a large velocity it "intends" to fix later —
except there is no later. The standard remedy is a **terminal cost**: an
extra state-only penalty at the horizon end, weighted by $W_{\text{term}}$, that
stands in for "the cost of all the steps beyond $N$". We reuse the same GN
state residual (no control term — there is no command at the terminal node),
scaled by $W_{\text{term}}$.

In [ ]:
W_TERM = 20.0
def term_cost_val(xv, ref):   rs, _ = _residuals(xv, u_hover, ref); return W_TERM*0.5*(rs@rs)
def term_cost_lx(xv, ref):    rs, _ = _residuals(xv, u_hover, ref); return W_TERM*(Jx_const.T @ rs)
def term_cost_lxx(xv, ref):   return W_TERM*(Jx_const.T @ Jx_const)

### 4.2 Acid test — analytic cost derivatives vs finite differences

We confirm the GN gradient $\ell_x=J^{\top}r$ matches the finite-difference
gradient of the true (wrapped) cost, at many points with modest angle errors (so
no test point straddles the $\pm\pi$ wrap seam). The GN Hessian $J^{\top}J$ is
exact for this quadratic-in-residual cost by construction.

In [ ]:
rng = np.random.default_rng(1)
def make_ref(p, ang, v, w, u):     # pack the 16-vector the cost expects
    return np.concatenate([p, ang, v, w, u]).astype(float)

worst_g = 0.0
for _ in range(120):
    xv = rng.normal(0, 0.3, 12); uv = u_hover + rng.normal(0, 1.5, 4)
    ref = make_ref(rng.normal(0,0.3,3), rng.normal(0,0.2,3), rng.normal(0,0.3,3),
                   rng.normal(0,0.2,3), u_hover)
    g_an = cost_lx(xv, uv, ref)
    g_fd = np.zeros(12); eps = 1e-6
    for j in range(12):
        dx = np.zeros(12); dx[j] = eps
        g_fd[j] = (cost(xv+dx, uv, ref) - cost(xv-dx, uv, ref)) / (2*eps)
    worst_g = max(worst_g, np.max(np.abs(g_an - g_fd)))
print(f"worst |analytic grad - finite-diff| over 120 points = {worst_g:.2e}")
assert worst_g < 1e-6, "cost gradient mismatch"
print("OK - cost derivatives match finite differences.")

## 5. Wiring the rocket into the iLQR solver

The generic solver lives in `../control/ilqr_ref.py` (the Python mirror of
`libs/control/ilqr.hpp`, conformance-tested against it in `../control/ilqr.ipynb`).
Here we only hand it the rocket's dynamics, Jacobians, the (identity) projection,
the stage/terminal cost, and the **per-input** actuator box. The solver returns
the optimal command sequence; we apply the first and recede.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../control'))
import ilqr_ref as _mpc

def _f(xv, uv):   return f_num(xv, uv)                  # continuous dynamics
def _jac(xv, uv): return fx_num(xv, uv), fu_num(xv, uv) # continuous Jacobians (RK4-chained inside)

# per-input actuator box (runtime PhysicsParams: F1/T1/T2/T3 min/max)
F1_MIN, F1_MAX = 0.0, 500.0
T_MIN,  T_MAX  = -10.0, 10.0
def _box():
    lo = np.array([F1_MIN, T_MIN, T_MIN, T_MIN])
    hi = np.array([F1_MAX, T_MAX, T_MAX, T_MAX])
    return lo, hi

u_ref = u_hover.copy()   # hover wrench [m g, 0, 0, 0]

def ref_at(refs, k):
    return refs if (isinstance(refs, np.ndarray) and refs.ndim == 1) else refs[k]

def _stage_cost(refs):                    # -> (val, lx, lxx, lu, luu), Gauss-Newton
    def sc(xv, uv, k):
        r = ref_at(refs, k)
        Jx, Ju = cost_Jx(xv, uv, r), cost_Ju(xv, uv, r)
        return cost(xv, uv, r), cost_lx(xv, uv, r), Jx.T@Jx, cost_lu(xv, uv, r), Ju.T@Ju
    return sc

def _term_cost(refs, N):                  # -> (val, lx, lxx), weighted by W_TERM
    def tc(xv):
        r = ref_at(refs, N)
        return term_cost_val(xv, r), term_cost_lx(xv, r), term_cost_lxx(xv, r)
    return tc

def ilqr(x0, refs, N, iters=60, tol=1e-6, us_init=None):
    lo, hi = _box()
    us0 = us_init if us_init is not None else [u_ref.copy() for _ in range(N)]
    return _mpc.ilqr(x0, _f, _jac, project, _stage_cost(refs), _term_cost(refs, N),
                     lo, hi, DT_MPC, us0, iters=iters, tol=tol)

### 5.1 Demo — stabilise from a perturbed state

We drop the rocket off-target and tilted, with a small angular rate, and ask the
MPC to bring it back to upright hover. The cost must decrease monotonically and
the commands must stay inside the box — the whole point of a *control-limited*
solver is that saturation is honoured *inside* the optimisation, not clipped
afterwards.

In [ ]:
ref0 = make_ref([0,0,0], [0,0,0], [0,0,0], [0,0,0], u_ref)   # upright hover setpoint
# a modest perturbation: the rocket is torque-limited (+/-10 N.m, I ~ 3.3 kg.m^2
# -> ~3 rad/s^2), so it can only tilt and recover so fast; the horizon (N*DT_MPC =
# 1.6 s) gives it room to translate back and settle upright.
x0 = np.array([0.6, -0.4, 0.4,  0.05, 0.04, 0.08,  0,0,0,  0.05, 0.0, 0.05], float)
xs, us, hist = ilqr(x0, ref0, N=80, iters=60)
U = np.array(us); X = np.array(xs)
lo, hi = _box()

pos_err = np.linalg.norm(X[-1,0:3]); att_err = np.degrees(np.linalg.norm(X[-1,3:6]))
print(f"cost {hist[0]:.1f} -> {hist[-1]:.1f} in {len(hist)-1} iterations")
print(f"final position error = {pos_err:.4f} m")
print(f"final attitude error = {att_err:.4f} deg")
assert np.all(U >= lo-1e-9) and np.all(U <= hi+1e-9), "box violated"
assert all(hist[i+1] <= hist[i]+1e-9 for i in range(len(hist)-1)), "cost not monotone"
assert pos_err < 0.10 and att_err < 5.0, "did not settle to upright hover"

t = np.arange(len(us))*DT_MPC
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].semilogy(hist, '-o', ms=3); ax[0].set_title("cost per iteration"); ax[0].set_xlabel("iter"); ax[0].grid(alpha=.3)
for i in range(3): ax[1].plot(t, X[:-1, i], label="xyz"[i])
ax[1].set_title("position [m]"); ax[1].set_xlabel("t [s]"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[2].plot(t, U[:, 0], label="F1 [N]")
ax2b = ax[2].twinx()
for i, lab in enumerate(["T1","T2","T3"]): ax2b.plot(t, U[:, i+1], '--', label=lab)
ax[2].set_title("wrench (F1 solid / torques dashed)"); ax[2].set_xlabel("t [s]")
ax[2].legend(loc='upper right'); ax2b.legend(loc='lower right', fontsize=8); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. Closed loop — receding horizon, and what MPC buys you

Now the honest comparison. From **one** trajectory optimisation we build three
controllers and fly them through the *true* plant (RK4 + a mid-flight gust):

- **(a) feedforward** — replay the planned commands open-loop;
- **(b) trajectory-LQR** — feedforward plus the time-varying feedback gains the
  backward pass already produces (a linear correction about the plan);
- **(c) MPC** — re-solve every step from the measured state, warm-started from
  the previous solution, and apply only the first command.

The reference is a smooth lateral+vertical move (a `smootherstep`, $C^2$ with
flat ends) with upright attitude; a lateral velocity gust hits mid-way.

In [ ]:
def smootherstep(a):   a = np.clip(a, 0, 1); return a*a*a*(a*(a*6-15)+10)
def smootherstep_d(a):
    a = np.clip(a, 0, 1); return 30*a*a*(a*(a-2)+1)

# N_h is the receding horizon -- matched to the C++ runtime's compile-time HORIZON
# (40). The move is deliberately gentle and slow: a torque-limited rocket tracks a
# fast lateral trajectory poorly, so a mild manoeuvre keeps the demo honest and the
# tracking tight.
SIM, N_h = 120, 40
tot = SIM + N_h + 1
tg  = np.arange(tot)*DT_MPC; T_move = SIM*DT_MPC
ph  = (tg/T_move - 0.15)/(0.80 - 0.15)
spr = smootherstep(ph); spd = smootherstep_d(ph)/((0.80-0.15)*T_move)
Pa, Pb = np.array([0,0,0.]), np.array([0.6, 0.4, 0.9])
p_ref = Pa + np.outer(spr, Pb - Pa)
v_ref = np.outer(spd, Pb - Pa)
ref_traj = np.array([make_ref(p_ref[k], [0,0,0], v_ref[k], [0,0,0], u_ref) for k in range(tot)])

G0, G1 = 50, 58
def gust(tk):
    gk = np.zeros(12)
    if G0 <= tk < G1: gk[6] = -0.05     # lateral (vx) velocity kick [m/s per step]
    return gk
x0 = np.zeros(12)

def plan_once():
    xs, us, _ = ilqr(x0, ref_traj[:SIM+1], SIM, iters=80)
    lo, hi = _box()
    _, Kg = _mpc.backward_pass(np.array(xs), list(us), _f, _jac,
                               _stage_cost(ref_traj[:SIM+1]), _term_cost(ref_traj[:SIM+1], SIM),
                               lo, hi, DT_MPC, 1e-6)
    return np.array(xs), list(us), Kg

def sim_apply(policy):
    xk = x0.copy(); Xh = [xk]; lo, hi = _box()
    for tk in range(SIM):
        uk = np.clip(policy(tk, xk), lo, hi)
        xk = F_d(xk, uk) + gust(tk); Xh.append(xk)
    return np.array(Xh)

def mpc_run(sim_steps=SIM, warm=True):
    xk = x0.copy(); Xh = [xk]; iters = []; us = [u_ref.copy() for _ in range(N_h)]; lo, hi = _box()
    for tk in range(sim_steps):
        _, us_sol, hist = ilqr(xk, ref_traj[tk:tk+N_h+1], N_h,
                               iters=8 if warm else 60, us_init=us if warm else None)
        iters.append(len(hist)-1)
        xk = F_d(xk, np.clip(us_sol[0], lo, hi)) + gust(tk); Xh.append(xk)
        us = us_sol[1:] + [us_sol[-1]]
    return np.array(Xh), np.array(iters)

In [ ]:
xs_p, us_p, Kg_p = plan_once()
Xa = sim_apply(lambda tk, xk: us_p[tk])                               # (a) feedforward
Xb = sim_apply(lambda tk, xk: us_p[tk] + Kg_p[tk] @ (xk - xs_p[tk]))  # (b) trajectory-LQR
Xc, it_c = mpc_run()                                                 # (c) MPC (warm)
_,  it_cold = mpc_run(sim_steps=15, warm=False)                      # cold, short: for the iter count

def track_err(Xh): return np.linalg.norm(Xh[:SIM+1, 0:3] - p_ref[:SIM+1], axis=1)
ea, eb, ec = track_err(Xa), track_err(Xb), track_err(Xc)
rms = {}
for name, e in [("(a) feedforward   ", ea), ("(b) trajectory-LQR", eb), ("(c) MPC           ", ec)]:
    rms[name.strip()] = np.sqrt((e**2).mean())
    print(f"{name}: RMS {np.sqrt((e**2).mean()):.3f} m, peak {e.max():.3f} m, post-gust settle {e[-1]:.3f} m")
print(f"warm start: {it_c.mean():.1f} iters/tick (peak {it_c.max()})  vs  cold {it_cold.mean():.1f}")
# feedback beats open-loop, and re-solving is cheap once warm-started
assert rms["(c) MPC"] <= rms["(a) feedforward"] and rms["(b) trajectory-LQR"] <= rms["(a) feedforward"]
assert it_c.mean() < it_cold.mean()

tt = np.arange(SIM+1)*DT_MPC
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(p_ref[:SIM+1,0], p_ref[:SIM+1,1], 'k--', lw=1, label='reference')
ax[0].plot(Xa[:,0], Xa[:,1], label='(a) FF')
ax[0].plot(Xb[:,0], Xb[:,1], label='(b) TVLQR')
ax[0].plot(Xc[:,0], Xc[:,1], label='(c) MPC')
ax[0].set_title('ground track x-y [m]'); ax[0].set_xlabel('x'); ax[0].set_ylabel('y'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(tt, ea, label='(a) FF'); ax[1].plot(tt, eb, label='(b) TVLQR'); ax[1].plot(tt, ec, label='(c) MPC')
ax[1].axvspan(G0*DT_MPC, G1*DT_MPC, color='r', alpha=.12, label='gust')
ax[1].set_title('position error [m]'); ax[1].set_xlabel('t [s]'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

**Reading the numbers honestly.** Feedback (b, c) beats open-loop (a) decisively
— the gust walks the feedforward run off-course while both closed-loop
controllers reject it. Between trajectory-LQR and MPC the gap for a *smooth*
disturbance is small: a good time-varying LQR about a feasible plan already does
most of the work. MPC's structural advantage shows up elsewhere — when the
*constraints* bite (commands riding the box) and re-optimising changes *which*
limits are active, or when the state leaves the region where the linearisation is
valid. That regime is explored, vehicle-agnostically, in `../control/ilqr.ipynb`;
here we keep the mild, honest run. The warm-start line shows why re-solving every
tick is affordable: a handful of iterations, not dozens, once seeded from the
previous solution.

## 7. C++ export — the prediction model for the runtime

Finally we emit the C++ the runtime consumes: `Dynamics(s,u,userF)` and
`Jacobians(s,u) -> f_x, f_u`, and **nothing else** — no control law, no gain.
The hand-written iLQR (`libs/control/ilqr.hpp`) supplies the algorithm; this
generated model supplies $f,f_x,f_u$. The generator is the shared
`MpcCodegen`; only the config (`rocket_mpc_config`, dims + Euler enum + the rocket
parameter set incl. the actuator box) is rocket-specific.

In [ ]:
import importlib
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('..'))
import mpc_codegen as mcg
importlib.reload(mcg)
import subprocess, tempfile

cfg = mcg.rocket_mpc_config()
# actuator-box symbols: solver-side bounds, absent from f but part of the param set
F1_max_s, F1_min_s, T1_max_s, T1_min_s, T2_max_s, T2_min_s, T3_max_s, T3_min_s = \
    sp.symbols('F1_max_s F1_min_s T1_max_s T1_min_s T2_max_s T2_min_s T3_max_s T3_min_s', real=True)
gen = (mcg.MpcCodegen(cfg)
       .set_state_symbols(state).set_input_symbols(ctrl)
       .set_physics_symbols([m, Ix, Iy, Iz, g, c, cz,
                             F1_max_s, F1_min_s, T1_max_s, T1_min_s, T2_max_s, T2_min_s, T3_max_s, T3_min_s])
       .set_user_force_symbols([uFx, uFy, uFz])
       .set_dynamics(f).set_jacobians(fx, fu))
hpath, cpath = gen.write()
print("emitted:\n ", hpath, "\n ", cpath)

### 7.1 Regression test — compile the C++ and compare to the notebook

Compile the emitted model with a tiny driver, evaluate `Dynamics` and `Jacobians`
at a random non-trivial point (with a non-zero external force), and check they
match `f_num`, `fx_num`, `fu_num` to machine precision — the §3.1 acid test, now
across the Python↔C++ boundary.

In [ ]:
rng = np.random.default_rng(7)
xs_t = rng.normal(0, 0.4, 12)
us_t = u_hover + rng.normal(0, 2.0, 4)
uF_t = np.array([0.7, -1.1, 0.3])

f_ref  = f_num(xs_t, us_t, uF_t)
fx_ref = fx_num(xs_t, us_t)
fu_ref = fu_num(xs_t, us_t)

driver = rf'''#include "{os.path.abspath(hpath)}"
#include <cstdio>
using M = CDS::Dynamics::{cfg.model_name};
int main() {{
  M mdl;
  M::StateVec   s  = {{{",".join(repr(float(v)) for v in xs_t)}}};
  M::InputVec   u  = {{{",".join(repr(float(v)) for v in us_t)}}};
  M::UserForces uF = {{{",".join(repr(float(v)) for v in uF_t)}}};
  auto dx = mdl.Dynamics(s, u, uF);
  for (double v : dx) printf("%.15g ", v); printf("\n");
  double fx[12][12], fu[12][4]; mdl.Jacobians(s, u, fx, fu);
  for (int i=0;i<12;i++) for(int j=0;j<12;j++) printf("%.15g ", fx[i][j]); printf("\n");
  for (int i=0;i<12;i++) for(int j=0;j<4;j++)  printf("%.15g ", fu[i][j]); printf("\n");
  return 0;
}}'''
d = tempfile.mkdtemp(); dp = os.path.join(d, "drv.cpp"); open(dp, "w").write(driver)
subprocess.run(["clang++","-std=c++20","-O2", dp, os.path.abspath(cpath), "-o", os.path.join(d,"drv")], check=True)
o = subprocess.run([os.path.join(d,"drv")], capture_output=True, text=True).stdout.split("\n")
f_c  = np.array(o[0].split(), float)
fx_c = np.array(o[1].split(), float).reshape(12, 12)
fu_c = np.array(o[2].split(), float).reshape(12, 4)
print("max|f  C++ - notebook| =", f"{np.max(np.abs(f_c  - f_ref)):.2e}")
print("max|fx C++ - notebook| =", f"{np.max(np.abs(fx_c - fx_ref)):.2e}")
print("max|fu C++ - notebook| =", f"{np.max(np.abs(fu_c - fu_ref)):.2e}")
assert max(np.max(np.abs(f_c-f_ref)), np.max(np.abs(fx_c-fx_ref)), np.max(np.abs(fu_c-fu_ref))) < 1e-9
print("REGRESSION OK - the exported C++ matches the notebook math.")

## 8. What we have, and what's left

We derived the Euler-angle 6-DOF rocket prediction model $f,f_x,f_u$, wrote a
Gauss–Newton tracking cost with direct (wrapped-heading) attitude errors and
per-input control weights, wired both into the shared control-limited iLQR
solver, and exported a C++ model that reproduces the notebook math to machine
precision. Closed-loop, the MPC tracks a smooth trajectory and rejects a gust,
with warm-started re-solves cheap enough to run every tick.

**Left to the runtime** (the C++ side, `core/Models/RocketMPC.{hpp,cpp}`): the
zero-order-hold at the `DT_MPC` cadence, the warm-start carried across ticks, the
true external force fed to the plant while the predictor assumes zero, and the
tracking cost weights exposed as tunable controller parameters. **Left to the
algorithm notebook** (`../control/ilqr.ipynb`): the derivation and the regimes
where MPC is decisive. The rocket-specific job — dynamics, cost, export — is
done.